In [ ]:
from google.colab import files
files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!pip install kaggle

In [ ]:
!kaggle datasets list

In [ ]:
!kaggle datasets download -d faisalahmed07/pneumonia-chest-xray-dataset

In [ ]:
import zipfile

with zipfile.ZipFile("pneumonia-chest-xray-dataset.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

In [ ]:
!ls dataset
!ls dataset/*

In [ ]:
import os

for root, dirs, files in os.walk("dataset"):
    print(root)

In [ ]:
# ✅ STEP 8: MERGE + LOAD DATASET (FINAL)

import os
import shutil
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_path = "dataset/Pneumonia_Dataset/train"
test_path  = "dataset/Pneumonia_Dataset/test"

# 🔥 Step 1: Merge Virus + Bacterial → PNEUMONIA
pneumonia_path = os.path.join(train_path, "PNEUMONIA")
os.makedirs(pneumonia_path, exist_ok=True)

# Move Virus images
virus_path = os.path.join(train_path, "Virus")
if os.path.exists(virus_path):
    for file in os.listdir(virus_path):
        shutil.move(os.path.join(virus_path, file), pneumonia_path)

# Move Bacterial images
bact_path = os.path.join(train_path, "Bacterial")
if os.path.exists(bact_path):
    for file in os.listdir(bact_path):
        shutil.move(os.path.join(bact_path, file), pneumonia_path)

print("✅ Merged classes into PNEUMONIA")

# 🔥 Step 2: Load Dataset (Binary Classification)
train_gen = ImageDataGenerator(rescale=1./255)
test_gen = ImageDataGenerator(rescale=1./255)

train_data = train_gen.flow_from_directory(
    train_path,
    target_size=(128,128),
    batch_size=32,
    class_mode='binary'
)

test_data = test_gen.flow_from_directory(
    test_path,
    target_size=(128,128),
    batch_size=32,
    class_mode='binary'
)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential()

# 🔹 Convolution Layers
model.add(Conv2D(32, (3,3), activation='relu', input_shape=(128,128,3)))
model.add(MaxPooling2D(2,2))

model.add(Conv2D(64, (3,3), activation='relu'))
model.add(MaxPooling2D(2,2))

# 🔹 Flatten
model.add(Flatten())

# 🔹 Fully Connected Layers
model.add(Dense(128, activation='relu'))
model.add(Dropout(0.5))  # helps improve accuracy

model.add(Dense(1, activation='sigmoid'))  # binary output

# 🔹 Compile
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
history = model.fit(
    train_data,
    epochs=5,
    validation_data=test_data
)

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import numpy as np
from tensorflow.keras.preprocessing import image

img = image.load_img("img7.jpg", target_size=(128,128))
img_array = image.img_to_array(img)/255.0
img_array = np.expand_dims(img_array, axis=0)

prediction = model.predict(img_array)

if prediction[0][0] > 0.5:
    print("❌ PNEUMONIA DETECTED")
else:
    print("✅ NORMAL")